
# Alcampo Web Scraper

This notebook collects product and price information from the Alcampo online store using **Selenium and BeautifulSoup**.

1. Runs the selected searches and loads additional results by scrolling.
2. Extracts product names, IDs, prices, unit prices, URLs, and collection timestamps.
3. Reviews the collected RAW data and checks collection quality.
4. Saves the latest RAW collection and a dated RAW historical snapshot.
5. Supports TEST_MODE without modifying any files.

Product deduplication, standardization, and historical price processing are handled by the central pipeline (`src/pipeline/`).

**Outputs:**
- `data/raw/alcampo_products_raw.csv`
- `data/history/alcampo/YYYY-MM-DD_raw.csv`


In [ ]:
import time
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, quote

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


HOME_URL = "https://www.compraonline.alcampo.es/"

SEARCH_TERMS = [
    "leche",
    "huevos",
    "manzana",
    "plátano",
    "tomate",
    "arroz",
    "pasta",
    "pan",
    "pollo",
    "carne picada",
    "salmón",
    "yogur",
    "queso",
    "agua",
    "zumo",
    "atún",
    "garbanzos",
    "aceite",
    "detergente",
    "papel higiénico",
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------

TEST_MODE = False
TEST_TERMS = ["leche"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

MAX_SCROLLS = 40
SCROLL_PAUSE = 1.5
STABLE_ROUNDS_TO_STOP = 3

print("Search terms:", TERMS_TO_SCRAPE)

Search terms: ['leche']


In [15]:

# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path
    if current.name == "scrapers":
        return current.parent
    raise FileNotFoundError("Project root not found. Run from the scrapers directory.")


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history" / "alcampo"

# Keep the RAW filename compatible with the central pipeline.
RAW_PATH = RAW_DIR / "alcampo_products_raw.csv"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"

print("Project root:", BASE_DIR)
print("Collection date:", RUN_DATE)
print("Current RAW:", RAW_PATH)
print("Historical RAW:", RAW_HISTORY_PATH)

Project root: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-13
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\alcampo_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\alcampo\2026-11-13_raw.csv


In [16]:
# --------------------------------------------------
# BROWSER AND NAVIGATION HELPERS
# --------------------------------------------------

from selenium.webdriver.chrome.options import Options
from selenium import webdriver

def start_alcampo_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    
    # Browser configuration
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=options)
    
    # Browser compatibility setting
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    driver.get(HOME_URL)
    return driver


def accept_alcampo_cookies(driver):
    xpaths = [
        "//button[contains(., 'Aceptar todas')]",
        "//button[contains(., 'Aceptar todo')]",
        "//button[contains(., 'Aceptar')]",
        "//button[contains(., 'Permitir todas')]",
    ]

    for xpath in xpaths:
        buttons = driver.find_elements(By.XPATH, xpath)

        for button in buttons:
            if button.is_displayed() and button.is_enabled():
                try:
                    button.click()
                    print("Cookies accepted.")
                    return
                except Exception:
                    pass

    print("No visible cookie banner.")


def assert_no_security_challenge(driver):
    body_text = driver.find_element(By.TAG_NAME, "body").text.lower()

    messages = [
        "captcha",
        "access denied",
        "confirma que no eres un robot",
        "comprueba que eres humano",
        "verify you are human",
    ]

    if any(message in body_text for message in messages):
        raise RuntimeError(
            "A security challenge was detected."
        )


def find_search_box(driver):
    # Try semantic attributes first.
    selectors = [
        'input[type="search"]',
        'input[placeholder*="Buscar en Alcampo"]',
        'input[placeholder*="Buscar"]',
        'input[aria-label*="Buscar"]',
        'input[name*="search"]',
        'input[id*="search"]',
    ]

    def visible_search_box():
        for selector in selectors:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)

            for element in elements:
                if element.is_displayed() and element.is_enabled():
                    return element

        return False

    try:
        return WebDriverWait(driver, 12).until(
            lambda _: visible_search_box()
        )

    except TimeoutException:
        diagnostics = []

        for element in driver.find_elements(By.TAG_NAME, "input"):
            if element.is_displayed():
                diagnostics.append({
                    "type": element.get_attribute("type"),
                    "name": element.get_attribute("name"),
                    "id": element.get_attribute("id"),
                    "placeholder": element.get_attribute("placeholder"),
                    "aria-label": element.get_attribute("aria-label"),
                })

        raise RuntimeError(
            "Could not find the Alcampo search input. "
            f"Visible inputs: {diagnostics}"
        )

In [17]:
# --------------------------------------------------
# PRODUCT SEARCH
# --------------------------------------------------

def count_product_links(driver):
    links = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/products/"]'
    )

    hrefs = {
        link.get_attribute("href")
        for link in links
        if link.get_attribute("href")
    }

    return len(hrefs)


def search_alcampo(driver, term):
    assert_no_security_challenge(driver)
    
    encoded_term = quote(term)
    # Use the website search URL
    search_url = f"https://www.compraonline.alcampo.es/search?q={encoded_term}"
    
    print(f"Opening search URL: {search_url}")
    driver.get(search_url)
    
    time.sleep(4)  # Allow time for the page to render
    assert_no_security_challenge(driver)
    
    # Report the current page for debugging
    print("Current page title:", driver.title)
    print("Product links detected:", count_product_links(driver))

    def search_loaded(d):
        assert_no_security_challenge(d)
        
        # Wait until product links appear
        new_count = count_product_links(d)
        if new_count > 0:
            return True
            
        return False

    try:
        WebDriverWait(
            driver,
            20,
            poll_frequency=0.4
        ).until(search_loaded)

    except TimeoutException:
        # A search can legitimately return no results.
        body_text = driver.find_element(By.TAG_NAME, "body").text.lower()

        if (
            "no hemos encontrado" in body_text
            or "sin resultados" in body_text
            or "0 resultados" in body_text
        ):
            print(f"No results for: {term}")
            return

        raise TimeoutException(
            f"Could not confirm the search page loaded for '{term}'."
        )

    print(
        f"Search completed: {term}\n"
        f"Current URL: {driver.current_url}"
    )

In [18]:
# --------------------------------------------------
# LOAD SEARCH RESULTS BY SCROLLING
# --------------------------------------------------


# --------------------------------------------------
# LOAD RESULTS PROGRESSIVELY
# --------------------------------------------------

def load_all_alcampo_products(driver, searched_item, max_steps=60):

    collected = {}
    no_new_at_bottom = 0

    def capture_visible_products():
        df = extract_alcampo_products(
            driver, searched_item, verbose=False
        )

        if df.empty:
            return 0

        before = len(collected)

        for record in df.to_dict("records"):
            url = record.get("url")
            if url:
                collected.setdefault(url, record)

        return len(collected) - before

    # Capture products initially visible
    capture_visible_products()

    print(f"Initial products collected: {len(collected)}")

    for step in range(1, max_steps + 1):

        assert_no_security_challenge(driver)

        # Scroll gradually instead of jumping to the bottom
        driver.execute_script(
            "window.scrollBy(0, Math.max(180, "
            "Math.floor(window.innerHeight * 0.6)));"
        )

        time.sleep(1.2)

        # Capture products after each scroll
        new_count = capture_visible_products()

        position, viewport, page_height = driver.execute_script(
            "return [window.scrollY, window.innerHeight, "
            "document.documentElement.scrollHeight];"
        )

        at_bottom = position + viewport >= page_height - 100

        if step == 1 or new_count or step % 10 == 0:
            print(
                f"Step {step}: +{new_count} products | "
                f"Total: {len(collected)}"
            )

        if at_bottom and new_count == 0:
            no_new_at_bottom += 1
        else:
            no_new_at_bottom = 0

        if no_new_at_bottom >= 6:
            print("End of results reached.")
            return pd.DataFrame(
                list(collected.values())
            ), "stable"

    print("Maximum scroll steps reached.")

    return pd.DataFrame(
        list(collected.values())
    ), "max_scrolls_reached"


In [19]:
# --------------------------------------------------
# DATA CLEANING HELPERS
# --------------------------------------------------

def clean_number(text):
    if text is None:
        return None

    match = re.search(
        r"\d+(?:[.,]\d+)?",
        str(text).replace("\xa0", " ")
    )

    if not match:
        return None

    return float(
        match.group()
        .replace(",", ".")
    )


def normalize_unit_name(unit_text):
    if not unit_text:
        return None

    unit = (
        unit_text
        .strip()
        .lower()
        .replace(".", "")
    )

    mappings = {
        "kg": "KILO",
        "kilo": "KILO",
        "kilogramo": "KILO",
        "kilogramos": "KILO",

        "l": "LITRO",
        "litro": "LITRO",
        "litros": "LITRO",

        "unidad": "UNIDAD",
        "ud": "UNIDAD",
        "uds": "UNIDAD",

        "docena": "DOCENA",
        "lavado": "LAVADO",
        "lavados": "LAVADO",
    }

    return mappings.get(unit, unit.upper())


def extract_price_per_unit(text):
    if not text:
        return None, None, None

    cleaned = (
        str(text)
        .replace("\xa0", " ")
        .strip()
    )

    # Typical format:
    # 1000ml(0,83 € por litro)
    match = re.search(
        r"(\d+(?:[.,]\d+)?)\s*€\s+por\s+"
        r"(litro|litros|kg|kilo|kilogramo|kilogramos|"
        r"unidad|ud|uds|docena|lavado|lavados)",
        cleaned,
        flags=re.IGNORECASE,
    )

    if match:
        value = float(
            match.group(1)
            .replace(",", ".")
        )

        unit = normalize_unit_name(
            match.group(2)
        )

        return value, unit, match.group(0)

    # Posibles formatos por 100 ml / 100 g
    match_100 = re.search(
        r"(\d+(?:[.,]\d+)?)\s*€\s*(?:por|/)\s*100\s*"
        r"(ml|g|gr)",
        cleaned,
        flags=re.IGNORECASE,
    )

    if match_100:
        value = float(
            match_100.group(1)
            .replace(",", ".")
        )

        measure = match_100.group(2).lower()

        if measure == "ml":
            return value * 10, "LITRO", match_100.group(0)

        return value * 10, "KILO", match_100.group(0)

    return None, None, None


def extract_product_id(url):
    if not url:
        return None

    match = re.search(
        r"/products/[^/]+/(\d+)",
        url
    )

    return match.group(1) if match else None

In [20]:
# --------------------------------------------------
# EXTRACT PRODUCTS FROM THE CURRENT PAGE
# --------------------------------------------------

def find_product_container(anchor):
    current = anchor

    for _ in range(8):
        current = current.parent

        if current is None:
            break

        text = current.get_text(" ", strip=True)

        # Product cards usually contain a price
        # and an add-to-cart button.
        if (
            "Precio" in text
            and "Añadir" in text
            and len(text) < 2500
        ):
            return current

    return anchor.parent


def extract_alcampo_products(driver, searched_item, verbose=True):
    soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )

    anchors = soup.select(
        'a[href*="/products/"]'
    )

    records = []
    seen_urls = set()
    scraped_at = datetime.now().isoformat(
        timespec="seconds"
    )

    for anchor in anchors:
        href = anchor.get("href")

        if not href:
            continue

        url = urljoin(HOME_URL, href)

        # The same product may have more than one link
        # inside its card.
        if url in seen_urls:
            continue

        seen_urls.add(url)

        container = find_product_container(anchor)

        if container is None:
            continue

        text = container.get_text(
            " ",
            strip=True
        )

        # --------------------------------------------------
        # PRODUCT NAME
        # --------------------------------------------------

        name = (
            anchor.get("aria-label")
            or anchor.get("title")
            or anchor.get_text(" ", strip=True)
        )

        # If the anchor has no usable text,
        # look for a title inside the product card.
        if not name:
            candidate = container.find(
                ["h2", "h3", "h4"]
            )

            if candidate:
                name = candidate.get_text(
                    " ",
                    strip=True
                )

        # --------------------------------------------------
        # PRODUCT PRICE
        # --------------------------------------------------

        price = None
        price_raw = None

        price_match = re.search(
            r"Precio\s+(\d+(?:[.,]\d+)?)\s*€",
            text,
            flags=re.IGNORECASE,
        )

        if price_match:
            price_raw = price_match.group(0)
            price = float(
                price_match.group(1)
                .replace(",", ".")
            )

        # --------------------------------------------------
        # PRODUCT PRICE POR UNIDAD
        # --------------------------------------------------

        price_per_unit, unit, ppu_raw = (
            extract_price_per_unit(text)
        )

        # --------------------------------------------------
        # PACKAGE SIZE
        # --------------------------------------------------

        quantity_raw = None

        quantity_match = re.search(
            r"\b\d+(?:[.,]\d+)?\s*"
            r"(?:ml|cl|l|g|gr|kg|uds?|unidades?)\b",
            text,
            flags=re.IGNORECASE,
        )

        if quantity_match:
            quantity_raw = quantity_match.group(0)

        product_id = extract_product_id(url)

        # Skip entries without required product information.
        
        if not name or price is None or price <= 0:
            continue

        records.append({
            "searched_item": searched_item,
            "supermarket": "Alcampo",
            "product_id": product_id,
            "name": name,
            "quantity_raw": quantity_raw,
            "price_raw": price_raw,
            "price": price,
            "price_per_unit_raw": ppu_raw,
            "price_per_unit": price_per_unit,
            "unit": unit,
            "url": url,
            "scraped_at": scraped_at,
        })

    df = pd.DataFrame(records)

    if verbose:
        print(f"Product links detected: {len(anchors)}")
        print(f"Products extracted: {len(df)}")

    return df

In [21]:
# --------------------------------------------------
# COLLECT PRODUCT DATA
# --------------------------------------------------

all_results = []
summary = []

driver = start_alcampo_driver()

try:
    time.sleep(3)
    accept_alcampo_cookies(driver)

    for i, term in enumerate(
        TERMS_TO_SCRAPE,
        start=1
    ):
        print(
            "\n" + "=" * 70
        )
        print(
            f"[{i}/{len(TERMS_TO_SCRAPE)}] "
            f"Searching: {term}"
        )

        start = time.monotonic()

        try:
            search_alcampo(
                driver,
                term
            )

            df_term, load_status = load_all_alcampo_products(driver, term)

            loaded_count = len(df_term)

            elapsed = round(
                time.monotonic() - start,
                2
            )

            all_results.append(
                df_term
            )

            summary.append({
                "searched_item": term,
                "status": load_status,
                "products_loaded": loaded_count,
                "rows_extracted": len(df_term),
                "seconds": elapsed,
            })

            print(
                f"✓ {len(df_term)} rows extracted "
                f"in {elapsed} s"
            )

            time.sleep(2)

        except Exception as exc:
            elapsed = round(
                time.monotonic() - start,
                2
            )

            print(
                f"ERROR for '{term}': "
                f"{type(exc).__name__}: {exc}"
            )

            summary.append({
                "searched_item": term,
                "status": "error",
                "products_loaded": None,
                "rows_extracted": 0,
                "seconds": elapsed,
            })

            # Stop after a search error; flag the run as partial.
            break

finally:
    driver.quit()


df_alcampo = (
    pd.concat(
        all_results,
        ignore_index=True
    )
    if all_results
    else pd.DataFrame()
)

df_summary = pd.DataFrame(
    summary
)

display(df_summary)

print(
    "\nTotal raw rows:",
    len(df_alcampo)
)

Cookies accepted.

[1/1] Searching: leche
Opening search URL: https://www.compraonline.alcampo.es/search?q=leche
Current page title: Resultados de la búsqueda para leche - Alcampo supermercado online
Product links detected: 5
Search completed: leche
Current URL: https://www.compraonline.alcampo.es/search?q=leche
Initial products collected: 5
Step 1: +5 products | Total: 10
Step 2: +5 products | Total: 15
Step 3: +5 products | Total: 20
Step 4: +5 products | Total: 25
Step 6: +5 products | Total: 30
Step 7: +5 products | Total: 35
Step 8: +5 products | Total: 40
Step 9: +3 products | Total: 43
Step 10: +2 products | Total: 45
Step 12: +4 products | Total: 49
Step 13: +5 products | Total: 54
Step 14: +5 products | Total: 59
Step 16: +5 products | Total: 64
Step 17: +5 products | Total: 69
Step 18: +5 products | Total: 74
Step 19: +5 products | Total: 79
Step 20: +0 products | Total: 79
Step 21: +5 products | Total: 84
Step 22: +5 products | Total: 89
Step 23: +5 products | Total: 94
Step

,searched_item,status,products_loaded,rows_extracted,seconds
0,leche,max_scrolls_reached,224,224,265.95



Total raw rows: 224


In [22]:
# --------------------------------------------------
# RAW DATA REVIEW
# --------------------------------------------------

if not df_alcampo.empty:
    display(
        df_alcampo.head(30)
    )

    print(
        "Rows without product_id:",
        df_alcampo["product_id"]
        .isna()
        .sum()
    )

    print(
        "Distinct product IDs:",
        df_alcampo["product_id"]
        .nunique()
    )

    print("\nRows per search:")

    display(
        df_alcampo
        .groupby("searched_item")
        .size()
        .rename("rows")
    )

else:
    print(
        "No data to review."
    )

,searched_item,supermarket,product_id,name,quantity_raw,price_raw,price,price_per_unit_raw,price_per_unit,unit,url,scraped_at
0,leche,Alcampo,54178,AUCHAN Leche entera de vaca 6 x 1 l Producto A...,1 l,"Precio 5,76 €",5.76,"0,96 € por litro",0.96,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-11-13T10:17:26
1,leche,Alcampo,54180,AUCHAN Leche semidesnatada de vaca 6 x 1l Prod...,1l,"Precio 5,28 €",5.28,"0,88 € por litro",0.88,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-11-13T10:17:26
2,leche,Alcampo,99193,L.R. Leche entera 6 x 1 l.,1 l,"Precio 5,70 €",5.70,"0,95 € por litro",0.95,LITRO,https://www.compraonline.alcampo.es/products/l...,2026-11-13T10:17:26
3,leche,Alcampo,53540,AUCHAN Leche entera de vaca 1 l. Producto Alcampo,1 l,"Precio 0,96 €",0.96,"0,96 € por litro",0.96,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-11-13T10:17:26
4,leche,Alcampo,53082,CENTRAL LECHERA ASTURIANA Leche de vaca semide...,1 l,"Precio 6,24 €",6.24,"1,04 € por litro",1.04,LITRO,https://www.compraonline.alcampo.es/products/c...,2026-11-13T10:17:26
5,leche,Alcampo,216880,AUCHAN Leche mini entera 6 x 200 ml Producto A...,200 ml,"Precio 1,71 €",1.71,"1,42 € por litro",1.42,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-11-13T10:17:28
6,leche,Alcampo,53004,CENTRAL LECHERA ASTURIANA Leche de vaca entera...,1l,"Precio 7,08 €",7.08,"1,18 € por litro",1.18,LITRO,https://www.compraonline.alcampo.es/products/c...,2026-11-13T10:17:28
7,leche,Alcampo,53096,PULEVA EcoLeche entera pack de 6 uds x 1 l.,6 uds,"Precio 8,26 €",8.26,"1,38 € por litro",1.38,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-11-13T10:17:28
8,leche,Alcampo,54434,PRÉSIDENT Leche entera de vaca de origen 100% ...,1 l,"Precio 6,43 €",6.43,"1,07 € por litro",1.07,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-11-13T10:17:28
9,leche,Alcampo,54497,PASCUAL Leche entera procedente de animales co...,1l,"Precio 7,68 €",7.68,"1,28 € por litro",1.28,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-11-13T10:17:28


Rows without product_id: 0
Distinct product IDs: 224

Rows per search:


searched_item
leche    224
Name: rows, dtype: int64

In [ ]:

# --------------------------------------------------
# VALIDATE AND SAVE CURRENT AND HISTORICAL RAW DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # --------------------------------------------------
    # VALIDATE COLLECTED DATA
    # --------------------------------------------------

    if df_alcampo.empty:
        raise RuntimeError(
            "No products collected. No files saved."
        )

    if df_summary.empty:
        raise RuntimeError(
            "No search results were recorded."
        )

    required_columns = [
        "searched_item",
        "supermarket",
        "product_id",
        "name",
        "quantity_raw",
        "price_raw",
        "price",
        "price_per_unit_raw",
        "price_per_unit",
        "unit",
        "url",
        "scraped_at",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df_alcampo.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing RAW columns: {missing_columns}"
        )

    # Check whether every expected search was attempted.
    all_terms_attempted = (
        len(df_summary) == len(SEARCH_TERMS)
        and set(df_summary["searched_item"]) == set(SEARCH_TERMS)
    )

    successful_searches = (
    df_summary["status"]
    .isin(["stable", "max_scrolls_reached"])
    .all()
)

    collection_complete = bool(
        all_terms_attempted and successful_searches
    )

    # Validate selling prices without altering RAW values.
    numeric_prices = pd.to_numeric(
        df_alcampo["price"],
        errors="coerce"
    )

    if (
        numeric_prices.isna().any()
        or (numeric_prices <= 0).any()
    ):
        raise RuntimeError(
            "Missing or invalid selling prices detected. "
            "No files saved."
        )

    # Validate product names.
    invalid_names = (
        df_alcampo["name"].isna()
        | df_alcampo["name"].astype(str).str.strip().eq("")
    )

    if invalid_names.any():
        raise RuntimeError(
            "Missing product names detected. No files saved."
        )

    # --------------------------------------------------
    # PROTECT EXISTING HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    if RAW_HISTORY_PATH.exists():
        raise FileExistsError(
            "A RAW historical snapshot already exists for "
            f"{RUN_DATE}: {RAW_HISTORY_PATH}. "
            "No files were modified."
        )

    # --------------------------------------------------
    # PREPARE RAW CSV FILES
    # --------------------------------------------------

    def prepare_csv_temp(frame, destination):
        destination = Path(destination)

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        fd, tmp_name = tempfile.mkstemp(
            prefix=".alcampo_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:
                frame.to_csv(file, index=False)

            return Path(tmp_name)

        except Exception:
            Path(tmp_name).unlink(missing_ok=True)
            raise

    # Only RAW files are produced by this notebook.
    outputs = [
        (RAW_HISTORY_PATH, df_alcampo),
        (RAW_PATH, df_alcampo),
    ]

    # --------------------------------------------------
    # SAVE RAW FILES
    # --------------------------------------------------

    staged = []

    try:
        # Prepare both files before updating destinations.
        for destination, frame in outputs:
            staged.append((
                prepare_csv_temp(frame, destination),
                Path(destination)
            ))

        # Save the historical snapshot first.
        # Exclusive creation prevents accidental overwriting.
        historical_tmp, historical_destination = staged[0]

        with historical_tmp.open("rb") as source:
            with historical_destination.open("xb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

        print("Saved historical RAW:", historical_destination)

        # Update the latest RAW only after history was saved.
        current_tmp, current_destination = staged[1]
        os.replace(current_tmp, current_destination)

        print("Saved current RAW:", current_destination)

    finally:
        for tmp, _ in staged:
            tmp.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    if collection_complete:
        print(
            "\nFull collection validated."
        )
    else:
        print(
            "\nWARNING: Partial collection. "
            "Available RAW products were saved."
        )

        display(
            df_summary.loc[
                df_summary["status"] != "stable",
                [
                    "searched_item",
                    "status",
                    "rows_extracted"
                ]
            ]
        )

        if not all_terms_attempted:
            print(
                "WARNING: Not all expected search terms "
                "were attempted."
            )

    print("\nCOLLECTION SUMMARY")
    print("Collection date:", RUN_DATE)
    print("Status:", "complete" if collection_complete else "partial")
    print("RAW rows:", len(df_alcampo))
    print("Distinct product IDs:", df_alcampo["product_id"].nunique())
    print("Searches attempted:", len(df_summary))
    print("Expected searches:", len(SEARCH_TERMS))

    print("\nSaved outputs:")
    print(RAW_PATH)
    print(RAW_HISTORY_PATH)

TEST MODE: no files will be saved.
